In [2]:
!pip install torch

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import torch
from diffusers import FluxTransformer2DModel

# Load just the transformer (the main model)
transformer = FluxTransformer2DModel.from_pretrained(
    "Qwen/Qwen-Image-Edit",
    subfolder="transformer",
    torch_dtype=torch.bfloat16
)

# View structure
print(transformer)

# Count parameters
total_params = sum(p.numel() for p in transformer.parameters())
print(f"\nTotal parameters: {total_params:,}")

/home/ubuntu/py312env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 9/9 [00:00<00:00, 52.93it/s]
Some weights of the model checkpoint at Qwen/Qwen-Image-Edit were not used when initializing FluxTransformer2DModel: 
 ['transformer_blocks.13.img_mod.1.weight, transformer_blocks.42.img_mlp.net.0.proj.weight, transformer_blocks.47.img_mod.1.weight, transformer_blocks.7.img_mod.1.weight, transformer_blocks.26.txt_mod.1.bias, transformer_blocks.37.img_mlp.net.0.proj.bias, transformer_blocks.51.img_mlp.net.2.weight, transformer_blocks.45.txt_mod.1.weight, transformer_blocks.56.txt_mlp.net.2.bias, transformer_blocks.39.txt_mlp.net.0.proj.bias, transformer_blocks.59.img_mlp.net.0.proj.weight, transformer_blocks.23.img_mlp.net.0.proj.weight, transformer_blocks.33.img_

FluxTransformer2DModel(
  (pos_embed): FluxPosEmbed()
  (time_text_embed): CombinedTimestepTextProjEmbeddings(
    (time_proj): Timesteps()
    (timestep_embedder): TimestepEmbedding(
      (linear_1): Linear(in_features=256, out_features=3072, bias=True)
      (act): SiLU()
      (linear_2): Linear(in_features=3072, out_features=3072, bias=True)
    )
    (text_embedder): PixArtAlphaTextProjection(
      (linear_1): Linear(in_features=768, out_features=3072, bias=True)
      (act_1): SiLU()
      (linear_2): Linear(in_features=3072, out_features=3072, bias=True)
    )
  )
  (context_embedder): Linear(in_features=3584, out_features=3072, bias=True)
  (x_embedder): Linear(in_features=64, out_features=3072, bias=True)
  (transformer_blocks): ModuleList(
    (0-59): 60 x FluxTransformerBlock(
      (norm1): AdaLayerNormZero(
        (silu): SiLU()
        (linear): Linear(in_features=3072, out_features=18432, bias=True)
        (norm): LayerNorm((3072,), eps=1e-06, elementwise_affine=Fals

In [1]:
import torch
from diffusers import QwenImageEditPipeline, PipelineQuantizationConfig

# Quantization config for diffusers
quantization_config = PipelineQuantizationConfig(
    quant_backend="bitsandbytes_4bit",
    quant_kwargs={"load_in_4bit": True, "bnb_4bit_compute_dtype": torch.bfloat16}
)

pipe = QwenImageEditPipeline.from_pretrained(
    "Qwen/Qwen-Image-Edit",
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16
)
pipe.enable_model_cpu_offload()

/home/ubuntu/py312env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading pipeline components...:  83%|████████▎ | 5/6 [00:09<00:02,  2.81s/it]`torch_dtype` is deprecated! Use `dtype` instead!

Loading pipeline components...: 100%|██████████| 6/6 [00:20<00:00,  3.46s/it]


In [2]:
print(pipe.vae)

AutoencoderKLQwenImage(
  (encoder): QwenImageEncoder3d(
    (nonlinearity): SiLU()
    (conv_in): QwenImageCausalConv3d(3, 96, kernel_size=(3, 3, 3), stride=(1, 1, 1))
    (down_blocks): ModuleList(
      (0-1): 2 x QwenImageResidualBlock(
        (nonlinearity): SiLU()
        (norm1): QwenImageRMS_norm()
        (conv1): QwenImageCausalConv3d(96, 96, kernel_size=(3, 3, 3), stride=(1, 1, 1))
        (norm2): QwenImageRMS_norm()
        (dropout): Dropout(p=0.0, inplace=False)
        (conv2): QwenImageCausalConv3d(96, 96, kernel_size=(3, 3, 3), stride=(1, 1, 1))
        (conv_shortcut): Identity()
      )
      (2): QwenImageResample(
        (resample): Sequential(
          (0): ZeroPad2d((0, 1, 0, 1))
          (1): Conv2d(96, 96, kernel_size=(3, 3), stride=(2, 2))
        )
      )
      (3): QwenImageResidualBlock(
        (nonlinearity): SiLU()
        (norm1): QwenImageRMS_norm()
        (conv1): QwenImageCausalConv3d(96, 192, kernel_size=(3, 3, 3), stride=(1, 1, 1))
        (

In [3]:
print(pipe.text_encoder)

Qwen2_5_VLForConditionalGeneration(
  (model): Qwen2_5_VLModel(
    (visual): Qwen2_5_VisionTransformerPretrainedModel(
      (patch_embed): Qwen2_5_VisionPatchEmbed(
        (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
      )
      (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-31): 32 x Qwen2_5_VLVisionBlock(
          (norm1): Qwen2RMSNorm((1280,), eps=1e-06)
          (norm2): Qwen2RMSNorm((1280,), eps=1e-06)
          (attn): Qwen2_5_VLVisionAttention(
            (qkv): Linear4bit(in_features=1280, out_features=3840, bias=True)
            (proj): Linear4bit(in_features=1280, out_features=1280, bias=True)
          )
          (mlp): Qwen2_5_VLMLP(
            (gate_proj): Linear4bit(in_features=1280, out_features=3420, bias=True)
            (up_proj): Linear4bit(in_features=1280, out_features=3420, bias=True)
            (down_proj): Linear4bit(in_features=3420, out_features=1280, bias=True)
        

In [4]:
print(pipe.transformer)

QwenImageTransformer2DModel(
  (pos_embed): QwenEmbedRope()
  (time_text_embed): QwenTimestepProjEmbeddings(
    (time_proj): Timesteps()
    (timestep_embedder): TimestepEmbedding(
      (linear_1): Linear4bit(in_features=256, out_features=3072, bias=True)
      (act): SiLU()
      (linear_2): Linear4bit(in_features=3072, out_features=3072, bias=True)
    )
  )
  (txt_norm): RMSNorm()
  (img_in): Linear4bit(in_features=64, out_features=3072, bias=True)
  (txt_in): Linear4bit(in_features=3584, out_features=3072, bias=True)
  (transformer_blocks): ModuleList(
    (0-59): 60 x QwenImageTransformerBlock(
      (img_mod): Sequential(
        (0): SiLU()
        (1): Linear4bit(in_features=3072, out_features=18432, bias=True)
      )
      (img_norm1): LayerNorm((3072,), eps=1e-06, elementwise_affine=False)
      (attn): Attention(
        (norm_q): RMSNorm()
        (norm_k): RMSNorm()
        (to_q): Linear4bit(in_features=3072, out_features=3072, bias=True)
        (to_k): Linear4bit(in_

In [5]:
print(pipe.scheduler)

FlowMatchEulerDiscreteScheduler {
  "_class_name": "FlowMatchEulerDiscreteScheduler",
  "_diffusers_version": "0.35.2",
  "base_image_seq_len": 256,
  "base_shift": 0.5,
  "invert_sigmas": false,
  "max_image_seq_len": 8192,
  "max_shift": 0.9,
  "num_train_timesteps": 1000,
  "shift": 1.0,
  "shift_terminal": 0.02,
  "stochastic_sampling": false,
  "time_shift_type": "exponential",
  "use_beta_sigmas": false,
  "use_dynamic_shifting": true,
  "use_exponential_sigmas": false,
  "use_karras_sigmas": false
}



In [8]:
from PIL import Image
import requests

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/cat.png"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB").resize((256, 256))

output = pipe(
    image=image,
    prompt="make the cat wear sunglasses",
    num_inference_steps=50
)

output

100%|██████████| 50/50 [02:59<00:00,  3.59s/it]


QwenImagePipelineOutput(images=[<PIL.Image.Image image mode=RGB size=1024x1024 at 0x796C0CA9E0F0>])

In [9]:
len(output.images)

1

In [10]:
output

QwenImagePipelineOutput(images=[<PIL.Image.Image image mode=RGB size=1024x1024 at 0x796C0CA9E0F0>])

In [11]:
!cat requirements.txt

accelerate==1.12.0
asttokens==3.0.1
async-lru==2.0.5
attrs==25.4.0
babel==2.17.0
bitsandbytes==0.48.2
bleach==6.3.0
certifi==2025.11.12
charset-normalizer==3.4.4
comm==0.2.3
debugpy==1.8.17
decorator==5.2.1
defusedxml==0.7.1
diffusers==0.35.2
executing==2.2.1
fastjsonschema==2.21.2
filelock==3.20.0
fqdn==1.5.1
fsspec==2025.10.0
h11==0.16.0
hf-xet==1.2.0
huggingface-hub==0.36.0
idna==3.11
importlib_metadata==8.7.0
ipykernel==7.1.0
ipython==9.7.0
ipython_pygments_lexers==1.1.1
jedi==0.19.2
Jinja2==3.1.6
json5==0.12.1
jsonpointer==3.0.0
jupyter_client==8.6.3
jupyter_core==5.9.1
jupyterlab_pygments==0.3.0
jupyterlab_widgets==3.0.16
lark==1.3.1
MarkupSafe==3.0.3
matplotlib-inline==0.2.1
mistune==3.1.4
mpmath==1.3.0
nest-asyncio==1.6.0
networkx==3.6
numpy==2.3.5
nvidia-cublas-cu12==12.8.4.1
nvidia-cuda-cupti-cu12==12.8.90
nvidia-cuda-nvrtc-cu12==12.8.93
nvidia-cuda-runtime-cu12==12.8.90
nvidia-cudnn-cu12==9.10.2.21
nvidia-cufft-cu12==11.3.3.83
nvidia-cufile-cu12==1.13.1.3
nvidia-curand-cu12=

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
